# RRT* Demo: Rewiring for Better Paths

This notebook teaches how RRT* improves on RRT by optimizing tree structure over time.

What you will learn:
- how RRT* chooses a better parent for each new node,
- how local rewiring lowers path cost,
- why RRT* is typically slower but yields better solutions.

Suggested exploration:
- increase `max_iters` and track path cost,
- compare behavior with different `neighbor_radius` values.

In [ ]:
%matplotlib inline\n
\n
import math\n
import random\n
\n
import matplotlib.pyplot as plt\n
import numpy as np

## 1. Build map and planning query

We use a map with narrow passages so rewiring has visible effects on route quality.

In [ ]:
H, W = 60, 60\n
grid = np.zeros((H, W), dtype=np.uint8)\n
\n
grid[10:50, 24] = 1\n
grid[10:50, 36] = 1\n
grid[30, 24:37] = 1\n
grid[14, 24] = 0\n
grid[46, 36] = 0\n
grid[30, 30] = 0\n
\n
start = (5.0, 5.0)\n
goal = (55.0, 55.0)

## 2. Shared geometry and collision checks

These helper functions are the same backbone used by both RRT and RRT*.

In [ ]:
def in_bounds(r, c):\n
    return 0 <= r < H and 0 <= c < W\n
\n
def free_point(p):\n
    r, c = int(round(p[0])), int(round(p[1]))\n
    return in_bounds(r, c) and grid[r, c] == 0\n
\n
def dist(a, b):\n
    return math.hypot(a[0] - b[0], a[1] - b[1])\n
\n
def line_free(a, b):\n
    n = int(max(abs(b[0] - a[0]), abs(b[1] - a[1]))) + 1\n
    for i in range(n + 1):\n
        t = i / max(1, n)\n
        p = (a[0] + t * (b[0] - a[0]), a[1] + t * (b[1] - a[1]))\n
        if not free_point(p):\n
            return False\n
    return True

## 3. Run RRT* with parent selection and rewiring

Core improvements over RRT in this cell:
- best-parent selection among nearby nodes,
- rewiring neighbors when a cheaper route is found.

Observation prompt:
- increasing `neighbor_radius` can improve quality but adds computation.

In [ ]:
rng = random.Random(29)\n
step_len = 2.0\n
neighbor_radius = 4.5\n
goal_radius = 2.5\n
goal_bias = 0.1\n
max_iters = 3500\n
\n
nodes = [start]\n
parent = {0: None}\n
cost = {0: 0.0}\n
goal_idx = None\n
\n
for _ in range(max_iters):\n
    if rng.random() < goal_bias:\n
        sample = goal\n
    else:\n
        sample = (rng.uniform(0, H - 1), rng.uniform(0, W - 1))\n
\n
    near_i = min(range(len(nodes)), key=lambda i: dist(nodes[i], sample))\n
    near = nodes[near_i]\n
    d = dist(near, sample)\n
    if d < 1e-9:\n
        continue\n
\n
    t = min(step_len / d, 1.0)\n
    new = (near[0] + t * (sample[0] - near[0]), near[1] + t * (sample[1] - near[1]))\n
    if not free_point(new):\n
        continue\n
\n
    nearby = [i for i, p in enumerate(nodes) if dist(p, new) <= neighbor_radius and line_free(p, new)]\n
    if near_i not in nearby:\n
        nearby.append(near_i)\n
\n
    best_parent = min(nearby, key=lambda i: cost[i] + dist(nodes[i], new))\n
    best_cost = cost[best_parent] + dist(nodes[best_parent], new)\n
\n
    nodes.append(new)\n
    ni = len(nodes) - 1\n
    parent[ni] = best_parent\n
    cost[ni] = best_cost\n
\n
    for i in nearby:\n
        cand = cost[ni] + dist(nodes[ni], nodes[i])\n
        if cand + 1e-9 < cost[i] and line_free(nodes[ni], nodes[i]):\n
            parent[i] = ni\n
            cost[i] = cand\n
\n
    if dist(new, goal) <= goal_radius and line_free(new, goal):\n
        if goal_idx is None:\n
            nodes.append(goal)\n
            goal_idx = len(nodes) - 1\n
        parent[goal_idx] = ni\n
        cost[goal_idx] = cost[ni] + dist(new, goal)\n
\n
print('Nodes:', len(nodes), 'Goal connected:', goal_idx is not None)

## 4. Recover final route and compute path cost

Backtracking through `parent` gives the final route. Cost reflects how well rewiring improved connectivity.

In [ ]:
path = []\n
if goal_idx is not None:\n
    cur = goal_idx\n
    while cur is not None:\n
        path.append(nodes[cur])\n
        cur = parent[cur]\n
    path.reverse()\n
\n
path_cost = 0.0\n
for i in range(len(path) - 1):\n
    path_cost += dist(path[i], path[i + 1])\n
\n
print('Path points:', len(path), 'Path cost:', round(path_cost, 2))

## 5. Visual analysis and comparison exercise

Try this quick comparison exercise:
1. Run with `max_iters = 800` and note path cost.
2. Run with `max_iters = 3500` and compare.
3. Contrast the result with plain RRT on the same map.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))\n
ax.imshow(grid, cmap='Greys', origin='upper')\n
\n
for i in range(1, len(nodes)):\n
    pi = parent.get(i)\n
    if pi is None:\n
        continue\n
    a, b = nodes[pi], nodes[i]\n
    ax.plot([a[1], b[1]], [a[0], b[0]], color='lightsteelblue', linewidth=0.5)\n
\n
if path:\n
    xs = [p[1] for p in path]\n
    ys = [p[0] for p in path]\n
    ax.plot(xs, ys, color='deepskyblue', linewidth=2.6, label='RRT* path')\n
\n
ax.scatter(start[1], start[0], c='limegreen', s=90, marker='o', label='start')\n
ax.scatter(goal[1], goal[0], c='crimson', s=90, marker='*', label='goal')\n
ax.set_title('RRT* with local rewiring')\n
ax.set_xticks([])\n
ax.set_yticks([])\n
ax.legend(loc='upper right')\n
plt.tight_layout()\n
plt.show()